In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

print("VAAC-Tiny Training Pipeline")
print("Step 195–201")

VAAC-Tiny Training Pipeline
Step 195–201


In [2]:
PROJECT_ROOT = Path(
    r"C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection"
)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed" / "CWRU"
WINDOW_DIR = PROCESSED_DIR / "windows"

print("Project root:")
print(PROJECT_ROOT)

print("\nWindow directory:")
print(WINDOW_DIR)

print("\nExists:", WINDOW_DIR.exists())

Project root:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection

Window directory:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows

Exists: True


In [7]:
TRAIN_METADATA = WINDOW_DIR / "train_metadata.csv"
VAL_METADATA = WINDOW_DIR / "validation_metadata.csv"
TEST_METADATA = WINDOW_DIR / "test_metadata.csv"

train_df = pd.read_csv(TRAIN_METADATA)
val_df = pd.read_csv(VAL_METADATA)
test_df = pd.read_csv(TEST_METADATA)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (390, 7)
Validation: (117, 7)
Test: (76, 7)


In [8]:
print("========== TRAIN METADATA ==========")
display(train_df.head())

print("\n========== COLUMNS ==========")
print(train_df.columns.tolist())

print("\n========== TRAIN CLASS DISTRIBUTION ==========")
print(train_df["class"].value_counts())

========== TRAIN METADATA ==========


,recording_id,source_file,signal_id,class,window_id,start_sample,end_sample
0,B007_3_X121,B007_3.mat,X121,Ball,0,0,12000
1,B007_3_X121,B007_3.mat,X121,Ball,1,6000,18000
2,B007_3_X121,B007_3.mat,X121,Ball,2,12000,24000
3,B007_3_X121,B007_3.mat,X121,Ball,3,18000,30000
4,B007_3_X121,B007_3.mat,X121,Ball,4,24000,36000



========== COLUMNS ==========
['recording_id', 'source_file', 'signal_id', 'class', 'window_id', 'start_sample', 'end_sample']

========== TRAIN CLASS DISTRIBUTION ==========
class
Healthy       276
Inner Race     57
Outer Race     38
Ball           19
Name: count, dtype: int64


In [9]:
print("TRAIN METADATA COLUMNS")
print("=" * 50)

for column in train_df.columns:
    print(f"\nColumn: {column}")
    print("Example:", train_df[column].iloc[0])

TRAIN METADATA COLUMNS

Column: recording_id
Example: B007_3_X121

Column: source_file
Example: B007_3.mat

Column: signal_id
Example: X121

Column: class
Example: Ball

Column: window_id
Example: 0

Column: start_sample
Example: 0

Column: end_sample
Example: 12000


In [10]:
possible_path_columns = []

for column in train_df.columns:
    values = train_df[column].astype(str)

    if values.str.contains(".npy", case=False, regex=False).any():
        possible_path_columns.append(column)

print("Possible window path columns:")
print(possible_path_columns)

Possible window path columns:
[]


In [11]:
print("First training metadata row:")
display(train_df.iloc[0])

First training metadata row:


recording_id    B007_3_X121
source_file      B007_3.mat
signal_id              X121
class                  Ball
window_id                 0
start_sample              0
end_sample            12000
Name: 0, dtype: object

In [12]:
recording_id = train_df.iloc[0]["recording_id"]
window_id = int(train_df.iloc[0]["window_id"])

print("Recording ID:", recording_id)
print("Window ID:", window_id)

Recording ID: B007_3_X121
Window ID: 0


In [13]:
train_window_dir = WINDOW_DIR / "train"

print("Training window directory:")
print(train_window_dir)

print("\nDirectory exists:")
print(train_window_dir.exists())

print("\nFirst 10 .npy files:")

for file in sorted(train_window_dir.glob("*.npy"))[:10]:
    print(file.name)

Training window directory:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\train

Directory exists:
True

First 10 .npy files:
B007_3_X121_window_0000.npy
B007_3_X121_window_0001.npy
B007_3_X121_window_0002.npy
B007_3_X121_window_0003.npy
B007_3_X121_window_0004.npy
B007_3_X121_window_0005.npy
B007_3_X121_window_0006.npy
B007_3_X121_window_0007.npy
B007_3_X121_window_0008.npy
B007_3_X121_window_0009.npy


In [14]:
def get_window_path(row, split):

    recording_id = str(row["recording_id"])
    window_id = int(row["window_id"])

    filename = f"{recording_id}_window_{window_id:04d}.npy"

    return WINDOW_DIR / split / filename

In [15]:
sample_path = get_window_path(
    train_df.iloc[0],
    "train"
)

print("Constructed path:")
print(sample_path)

print("\nExists:", sample_path.exists())

Constructed path:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\train\B007_3_X121_window_0000.npy

Exists: True


In [16]:
sample_window = np.load(sample_path)

print("Shape:", sample_window.shape)
print("Data type:", sample_window.dtype)
print("Minimum:", sample_window.min())
print("Maximum:", sample_window.max())
print("Mean:", sample_window.mean())
print("Standard deviation:", sample_window.std())

Shape: (12000,)
Data type: float64
Minimum: -0.6096190419161677
Maximum: 0.6239133333333333
Mean: 0.003428179876912841
Standard deviation: 0.1547666237361117


In [17]:
row = train_df.iloc[0]

expected_length = int(row["end_sample"]) - int(row["start_sample"])

print("Start sample:", row["start_sample"])
print("End sample:", row["end_sample"])
print("Expected samples:", expected_length)
print("Actual samples:", len(sample_window))

Start sample: 0
End sample: 12000
Expected samples: 12000
Actual samples: 12000


In [18]:
for i in range(5):

    row = train_df.iloc[i]

    path = get_window_path(
        row,
        "train"
    )

    print(
        f"Window {i}:",
        path.name,
        "| Exists:",
        path.exists()
    )

Window 0: B007_3_X121_window_0000.npy | Exists: True
Window 1: B007_3_X121_window_0001.npy | Exists: True
Window 2: B007_3_X121_window_0002.npy | Exists: True
Window 3: B007_3_X121_window_0003.npy | Exists: True
Window 4: B007_3_X121_window_0004.npy | Exists: True


In [20]:
CLASS_NAMES = [
    "Healthy",
    "Ball",
    "Inner Race",
    "Outer Race"
]

CLASS_TO_INDEX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

train_df["label"] = train_df["class"].map(CLASS_TO_INDEX)
val_df["label"] = val_df["class"].map(CLASS_TO_INDEX)
test_df["label"] = test_df["class"].map(CLASS_TO_INDEX)

In [22]:
X_train, y_train = load_windows(
    train_df,
    "train"
)

X_val, y_val = load_windows(
    val_df,
    "validation"
)

X_test, y_test = load_windows(
    test_df,
    "test"
)

In [23]:
def load_windows(metadata_df, split):

    X = []
    y = []

    for _, row in metadata_df.iterrows():

        path = get_window_path(row, split)

        if not path.exists():
            raise FileNotFoundError(
                f"Window file not found:\n{path}"
            )

        signal = np.load(path)

        if signal.shape != (12000,):
            raise ValueError(
                f"Unexpected window shape {signal.shape}\n"
                f"File: {path}"
            )

        X.append(signal.astype(np.float32))
        y.append(int(row["label"]))

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    return X, y

In [24]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (390, 12000)
y_train: (390,)
X_val: (117, 12000)
y_val: (117,)
X_test: (76, 12000)
y_test: (76,)


In [25]:
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("Final CNN input shapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

Final CNN input shapes:
X_train: (390, 12000, 1)
X_val: (117, 12000, 1)
X_test: (76, 12000, 1)
